In [1]:
import os 
import sys 
import pandas as pd 
from pathlib import Path
from dataclasses import dataclass


In [2]:
#Entity Configuration 
@dataclass
class DataIngestionConfig:
    root_dir : Path 
    source_url : str 
    user_name : str 
    password : str 
    local_data_file : Path

In [3]:
pwd

'c:\\Users\\RSR\\PYTHON\\ServicenowGamification\\Research'

In [4]:
os.chdir("c:\\Users\\RSR\\PYTHON\\ServicenowGamification\\src")

In [5]:
pwd

'c:\\Users\\RSR\\PYTHON\\ServicenowGamification\\src'

In [6]:
from ServicenowGamification.Constant import *
from ServicenowGamification.Utils.Common import *
import requests

In [7]:
os.chdir("c:\\Users\\RSR\\PYTHON\\ServicenowGamification")

In [8]:
class ConfigurationManager:
    def __init__(self,
                 CONFIG_PATH=CONFIG_FILE_PATH,
                 PARAMS_PATH=PARAMS_FILE_PATH

                 ):
         print(CONFIG_PATH)
         #ROOT_DIR = ROOT_DIR = Path.cwd()
                 # Read configuration files
         #print("ROOT_DIR :", ROOT_DIR)
         
         #config_path = ROOT_DIR / "Config" / "Config.yaml"
         #params_path = ROOT_DIR / "Params.yaml"

        
         #self.config=Read_Yaml(Path("Config/Config.yaml"))
         print(f"path config yaml {Path(CONFIG_PATH)}")
         self.config=Read_Yaml(Path(CONFIG_PATH))
         self.params=Read_Yaml(Path(PARAMS_PATH))
         self.Data_Ingestion = self.config.Data_Ingestion
        
    def get_data_ingestion_config(self)->DataIngestionConfig:
        Config=self.Data_Ingestion
        data_ingestion_config=DataIngestionConfig(
            root_dir=Config.root_dir,
            source_url=Config.source_url,
            user_name=Config.user_name,
            password=Config.password,
            local_data_file=Config.local_data_file

        )

        return data_ingestion_config



In [ ]:
import os
from pathlib import Path
os.chdir(r"C:\Users\RSR\PYTHON\ServicenowGamification")
import pandas as pd
import requests
from ServicenowGamification.Config.ConfigurationManager import *
from ServicenowGamification.Utils.Common import *
from ServicenowGamification.Entity.Config_Entity import *




class DataIngestion:
    def __init__(self,config:DataIngestionConfig):
        self.config=config
        create_directories([self.config.root_dir])
        
    def LoadIncidents(self):
        INSTANCE = self.config.source_url
        USERNAME = self.config.user_name
        PASSWORD = self.config.password
        # print("INSTANCE",INSTANCE)
        # print("USERNAME",USERNAME)
        # print("PASSWORD",PASSWORD)

        # INSTANCE = "https://dev394794.service-now.com"
        # USERNAME = "admin"
        # PASSWORD = "TyMdk!/WQa00"

        url = f"{INSTANCE}/api/now/table/incident"

        params = {
            "sysparm_limit": 100,
            "sysparm_display_value": "true"
        }

        response = requests.get(
            url,
            auth=(USERNAME, PASSWORD),
            headers={"Accept": "application/json"},
            params=params
        )

        response.raise_for_status()
        print(f"\n*-----------------------------Service Now Connection ----------------------------------------*\n")
        print("ServiceNow connection established successfully!")
        
        incidents = response.json()["result"]

        

        df = pd.json_normalize(incidents)
        print(type(incidents))
        print(type(df))
        local_path = self.config.local_data_file
        df.to_csv(local_path, index=False)
        print(f"\n*---------------------------- Data Frame Total Incidents ------------------------------------*\n")
        print("Data Frame : Total Incidents:", len(df))
        print(f"\n*-------------------------- Data Frame Total Columns or Features ----------------------------*\n")

        #print(f"Data Frame : Columns/Features Details: {df.columns.tolist()}")
        print(f"\n*-------------------------- Data Frame Share(Row,Clumns) ------------------------------------*\n")
        
        print(f"Data Frame Shape (Rows, Columns): {df.shape}")

        print(f"\n*------------------------------ Data Frame Sample Data -------------------------------------*\n")
        

        features = [

            "number",
            "short_description",
            "description",
            "state",
            "priority",
            "category",
            "assigned_to"]

        #print(df[features].head(2))

    def LoadIncidents_all(self):

        try:

            # ========================================================
            # CONFIGURATION
            # ========================================================

            INSTANCE = self.config.source_url
            USERNAME = self.config.user_name
            PASSWORD = self.config.password

            url = (
                f"{INSTANCE}"
                "/api/now/table/incident"
            )

            # ========================================================
            # PAGINATION
            # ========================================================

            limit = 100
            offset = 0
            all_incidents = []
            while True:
                print(
                    f"\nFetching records "
                    f"from {offset} to {offset + limit}"
                )
                params = {

                    # Newest first
                    "sysparm_query":
                        "ORDERBYDESCopened_at",

                    "sysparm_limit":
                        limit,

                    "sysparm_offset":
                        offset,

                    "sysparm_display_value":
                        "true"
                }

                # ====================================================
                # API CALL
                # ====================================================

                response = requests.get(

                    url,
                    auth=(
                        USERNAME,
                        PASSWORD
                    ),
                    headers={
                        "Accept":
                            "application/json"
                    },
                    params=params,
                    timeout=60
                )
                response.raise_for_status()
                incidents = response.json().get(
                    "result",
                    []
                )
                print(
                    "Records received:",
                    len(incidents)
                )

                # ====================================================
                # STOP CONDITION
                # ====================================================
                if not incidents:
                    break

                # ====================================================
                # ADD RECORDS
                # ====================================================

                all_incidents.extend(
                    incidents
                )

                # ====================================================
                # LAST PAGE
                # ====================================================

                if len(incidents) < limit:
                    break

                offset += limit

            # ========================================================
            # DATAFRAME
            # ========================================================

            df = pd.json_normalize(
                all_incidents
            )

            print(
                "\n"
                "*-----------------------------------------*"
            )

            print(
                "TOTAL INCIDENTS:",
                len(df)
            )

            print(
                "DATAFRAME SHAPE:",
                df.shape
            )

            # ========================================================
            # SORT NEWEST FIRST
            # ========================================================

            if "opened_at" in df.columns:

                df["opened_at"] = pd.to_datetime(
                    df["opened_at"],
                    errors="coerce"
                )

                df = df.sort_values(
                    "opened_at",
                    ascending=False
                )

            # ========================================================
            # SHOW RECENT INCIDENTS
            # ========================================================

            print(
                "\nLatest 20 incidents:"
            )

            columns = [
                "number",
                "opened_at",
                "state",
                "priority",
                "assigned_to"
            ]

            available_columns = [
                col
                for col in columns
                if col in df.columns
            ]

            print(
                df[
                    available_columns
                ].head(20)
            )

            # ========================================================
            # SAVE
            # ========================================================

            local_path = (
                self.config.local_data_file
            )

            df.to_csv(
                local_path,
                index=False
            )

            print(
                "\nData saved to:",
                local_path
            )
            df['assigned_to']=df['assigned_to.display_value']
            return df
        
        except Exception as e:

            print(
                "ServiceNow Data Ingestion Failed:"
            )

            print(e)

            raise e



        return df 

In [10]:
config = ConfigurationManager().get_data_ingestion_config()

ingestion = DataIngestion(config)

df = ingestion.LoadIncidents_all()



Reading YAML : C:\Users\RSR\PYTHON\ServicenowGamification\Config\Config.yaml
[2026-08-13 21:24:50,805: INFO: Common: YAML file loaded successfully : C:\Users\RSR\PYTHON\ServicenowGamification\Config\Config.yaml]


Reading YAML : C:\Users\RSR\PYTHON\ServicenowGamification\Params.yaml
[2026-08-13 21:24:50,842: INFO: Common: YAML file loaded successfully : C:\Users\RSR\PYTHON\ServicenowGamification\Params.yaml]
Found YAML: C:\Users\RSR\PYTHON\ServicenowGamification\dvc.yaml
Found YAML: C:\Users\RSR\PYTHON\ServicenowGamification\params.yaml
Found YAML: C:\Users\RSR\PYTHON\ServicenowGamification\Config\Config.yaml
[2026-08-13 21:24:50,956: INFO: Common: Created directory : Artifacts]
[2026-08-13 21:24:50,964: INFO: Common: Created directory : Artifacts/DataIngestion]
[2026-08-13 21:24:50,964: INFO: Common: Created directory : Artifacts/DataIngestion]

Fetching records from 0 to 100
Records received: 79

*-----------------------------------------*
TOTAL INCIDENTS: 79
DATAFRAME SHAPE: (79, 110)

Latest 20 incidents:
        number           opened_at        state      priority assigned_to
0   INC0010020 2026-08-09 07:22:36      On Hold  3 - Moderate         NaN
1   INC0010019 2026-08-09 07:07:35  In Pr